# Rebaseline Smoke (Notebook-First)
Notebook này dùng để test/trial trực quan cho Phase 5 thay vì chạy lệnh terminal dài.

In [1]:
from pathlib import Path
import polars as pl
from src.feature_config import FeatureConfig
from src.feature_engineering import FeaturePipeline
from src.rebaseline import RebaselineRunner, RebaselineConfig

ModuleNotFoundError: No module named 'src'

In [ ]:
# Tạo sample input cho smoke run
root = Path('data/processed')
sample_dir = Path('data/features/smoke')
sample_dir.mkdir(parents=True, exist_ok=True)
train_src = root / 'lichess_2025-12_ml.parquet'
val_src = root / 'lichess_2026-01_ml.parquet'
train_sample = sample_dir / 'train_sample_input.parquet'
val_sample = sample_dir / 'val_sample_input.parquet'
pl.scan_parquet(str(train_src)).head(3000).collect().write_parquet(train_sample)
pl.scan_parquet(str(val_src)).head(3000).collect().write_parquet(val_sample)
train_sample, val_sample

In [ ]:
# Chạy pipeline FE trên sample
cfg = FeatureConfig(
    batch_size=1000,
    train_features_file=sample_dir / 'train_features.parquet',
    val_features_file=sample_dir / 'val_features.parquet',
    tfidf_vocab_file=sample_dir / 'tfidf_vocabulary.pkl',
    svd_components_file=sample_dir / 'svd_components.pkl',
    feature_columns_file=sample_dir / 'feature_columns.json',
)
pipe = FeaturePipeline(config=cfg)
train_stats = pipe.process_split_to_parquet([str(train_sample)], cfg.train_features_file, 'train')
val_stats = pipe.process_split_to_parquet([str(val_sample)], cfg.val_features_file, 'val')
train_df = pl.read_parquet(cfg.train_features_file)
val_df = pl.read_parquet(cfg.val_features_file)
pipe.verify_schema_consistency(train_df, val_df)
pipe.save_metadata(train_df.columns)
train_stats, val_stats

In [ ]:
# Re-baseline + ablation + artifact
runner = RebaselineRunner(feature_config=cfg, model_config=RebaselineConfig(n_estimators=80, max_depth=6))
summary = runner.run_full_rebaseline()
runner.save_feature_importance_plot(summary['full']['top_feature_importance'], sample_dir / 'feature_importance_top20.png')
runner.save_summary_json(summary, sample_dir / 'rebaseline_summary.json')
summary['full']['accuracy'], summary['full']['macro_f1'], summary['ablation']

## Kỳ vọng
- Sinh `data/features/smoke/feature_importance_top20.png`
- Sinh `data/features/smoke/rebaseline_summary.json`
- Dùng kết quả để cập nhật Task 5.3, 5.5, 5.6, 5.7